# 07 · Parallel checks and a join

ParallelAgent runs independent branches concurrently. A surrounding SequentialAgent waits for all branches before summarizing. Give branches distinct state keys.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 5 Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import SequentialAgent, ParallelAgent
stock = llm("stock_check", "Use get_product for P101; report available stock.", tools=[get_product], output_key="stock_result")
customer = llm("customer_check", "Use get_customer for C101; report membership tier.", tools=[get_customer], output_key="customer_result")
checks = ParallelAgent(name="parallel_checks", sub_agents=[stock, customer])
join = llm("join_results", "Summarize stock: {stock_result}; customer: {customer_result}. Do not place an order.")
pipeline = SequentialAgent(name="pre_order_checks", sub_agents=[checks, join])
lab = await make_lab(pipeline)
await ask(lab, "Can customer C101 buy two P101 hubs?")


## Try it

Add an independent shipping-policy branch. Compare latency and total model calls: parallelism saves waiting time, not tokens.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
